First Choosing the subject and the date<br>
   * The Subject : Analyzing Customer Reviews for Sentiment and tweet Recommendation.<br>
   * The Data from : https://www.kaggle.com/datasets/kazanova/sentiment140<br>
   
     It contains the following 6 fields:<br>
      target: the polarity of the tweet (0 = negative, 2 = neutral, 4 = positive)<br>
      ids: The id of the tweet ( 2087)<br>
      date: the date of the tweet (Sat May 16 23:58:44 UTC 2009)<br>
      flag: The query (lyx). If there is no query, then this value is NO_QUERY.<br>
      user: the user that tweeted (robotickilldozr)<br>
      text: the text of the tweet (Lyx is cool)<br>



In [4]:
import pandas as pd
import numpy as np
import re
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score

C:\Users\Public\Documents\Wondershare\CreatorTemp\ipykernel_6420\4040783014.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


load the data from Kaddle wbsite , Dataset columns and Extract relevant columns <br>
1-target (0: negative, 4 change to 1: positive), 2-text (tweet)

In [5]:
# helper function to Remove the url ftom the tweet 
def clean_tweet(text):
    text = re.sub(r"http\S+|www\S+", '', text)
    text = text.lower()
    return text

In [7]:
df = pd.read_csv('training.1600000.processed.noemoticon.csv', encoding='ISO-8859-1', header=None)
df.columns = ['target', 'id', 'date', 'flag', 'user', 'text']

# Convert 4 to 1 (positive/negative on another wise (0 0r 1) )
df['target'] = df['target'].replace(4, 1) 

X = df['text'].apply(clean_tweet).values  # Feature   both of them is np_array
y = df['target'].values  # Labels  (0 or 1)

print("The Feature X: ", X ,"\n")
print("The Labels y ",y)

The Feature X:  ["@switchfoot  - a that's a bummer.  you shoulda got david carr of third day to do it. ;d"
 "is upset that he can't update his facebook by texting it... and might cry as a result  school today also. blah!"
 '@kenichan i dived many times for the ball. managed to save 50%  the rest go out of bounds'
 ... 'are you ready for your mojo makeover? ask me for details '
 'happy 38th birthday to my boo of alll time!!! tupac amaru shakur '
 'happy #charitytuesday @thenspcc @sparkscharity @speakinguph4h '] 

The Labels y  [0 0 0 ... 1 1 1]


Raw text data needs to be converted into a numerical format because machine learning algorithms typically work with numerical input

In [41]:

vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
X_tfidf = vectorizer.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)
base_model = LogisticRegression(multi_class='auto', solver='lbfgs', max_iter=1000)



Part two: <br>
1) Cross Validation - train five models each time on another 80% of the training data

In [6]:
# 80 from data is training data , 20 from data is teating data 
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
scores = cross_val_score(base_model, X_train, y_train, cv=cv)
print ("Cross-validation scores", scores)
print("Average score: ", np.mean(scores))

Cross-validation scores [0.77185547 0.77191406 0.77333203 0.77175781 0.77250781]
Average score:  0.7722734375


In [16]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
models = []
scores = []
for train_index, val_index in kf.split(X_train):
    # Split data into training and validation sets for this fold
    X_train_fold, X_valia_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_valia_fold = y_train[train_index], y_train[val_index]
    
    # Train the model on the training fold
    model = base_model.fit(X_train_fold, y_train_fold)
    # Validate the model on the validation fold
    y_pred = model.predict(X_valia_fold)
    
    # Evaluate the model and save it
    score = accuracy_score(y_valia_fold, y_pred)
    models.append(model)
    scores.append(score)

# Print the performance of the models
print("Cross-validation scores: ", scores)
print("Average score: ", np.mean(scores))

Cross-validation scores:  [0.77264453125, 0.77103515625, 0.7722578125, 0.77165625, 0.77298828125]
Average score:  0.77211640625


2) As in 1, only draw 50% of the data each time (without remembering which models we drew) - 5 models again

In [8]:
# 50 from data is training data , 50 from data is teating data
cv = ShuffleSplit(n_splits=5, test_size=0.5, random_state=42) 
scores = cross_val_score(base_model, X_train, y_train, cv=cv)
print ("Cross-validation scores", scores)
print("Average score: ", np.mean(scores))

Cross-validation scores [0.77151875 0.7713625  0.77110469 0.77092969 0.77145781]
Average score:  0.7712746875


In [43]:
# 50 from data is training data , 50 from data is teating data
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.5, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
models = []
scores = []
for train_index, val_index in kf.split(X_train):
    # Split data into training and validation sets for this fold
    X_train_fold, X_valia_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_valia_fold = y_train[train_index], y_train[val_index]
    
    # Train the model on the training fold
    model = base_model.fit(X_train_fold, y_train_fold)
    # Validate the model on the validation fold
    y_pred = model.predict(X_valia_fold)
    
    # Evaluate the model and save it
    score = accuracy_score(y_valia_fold, y_pred)
    models.append(model)
    scores.append(score)

# Print the performance of the models
print("Cross-validation scores: ", scores)
print("Average score: ", np.mean(scores))

Cross-validation scores:  [0.77145, 0.7706875, 0.7702125, 0.76994375, 0.77168125]
Average score:  0.7707949999999999


3) adaboost - better written by you - also here five models

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Initialize alphas and models
alphas = []
models = []
n_estimators = 5  # AdaBoost algorithm will train 5 weak classifiers

n_samples = X_train.shape[0]  
weights = np.ones(n_samples) / n_samples  

for i in range(n_estimators):
    model = DecisionTreeClassifier(max_depth=1)  
    model.fit(X_train, y_train, sample_weight=weights)
    y_pred = model.predict(X_train)
    
    # Compute error
    error = np.sum(weights * (y_pred != y_train)) / np.sum(weights)
    
   
    if error > 0.5:
        break
    if error == 0:
        alpha = 1
    else:
        alpha = 0.5 * np.log((1 - error) / (error + 1e-10))
    
    
    alphas.append(alpha)
    models.append(model)
    
    # Update sample weights
    weights *= np.exp(alpha * (y_pred != y_train))
    weights /= np.sum(weights)  # Normalize weights

print("Alphas:", alphas)


H_model = np.zeros(X_test.shape[0])
for alpha, model in zip(alphas, models):
    H_model += alpha * model.predict(X_test) 

# Convert H_model to class predictions (1 or -1)
H_model_pred = np.sign(H_model)
print("Ensemble Model Accuracy:", accuracy_score(y_test, H_model_pred))


In [ ]:
# 80 from data is training data , 20 from data is testing data 
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
models = []
scores = []
f1Scores = []
for train_index, val_index in kf.split(X_train):
    # Split data into training and validation sets for this fold
    X_train_fold, X_valia_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_valia_fold = y_train[train_index], y_train[val_index]
    
    # Train the model on the training fold
    model = base_model.fit(X_train_fold, y_train_fold)
    # Validate the model on the validation fold
    y_pred = model.predict(X_valia_fold)
    
    # Evaluate the model and save it
    score = accuracy_score(y_valia_fold, y_pred)
    f1 = f1_score(y_valia_fold, y_pred)
    
    models.append(model)
    scores.append(score)
    f1Scores.append(f1)

# Print the performance of the models
print("Cross-validation scores: ", scores)
print("Average score: ", np.mean(scores))

# print("f1 Cross-validation scores: ", f1Scores)
# print("Average score: ", np.mean(f1Scores))